In [1]:
import torch, json, os, tqdm
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import torchvision.models as models

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode

# Dataset

In [3]:
augmentation_pipeline_train = transforms.Compose([
    transforms.RandomChoice([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.HAMMING, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.HAMMING, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.LANCZOS, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.LANCZOS, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.BOX, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.BOX, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.NEAREST, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.NEAREST, antialias=True),
        transforms.Resize((224, 224), interpolation=InterpolationMode.NEAREST_EXACT, antialias=False),
        transforms.Resize((224, 224), interpolation=InterpolationMode.NEAREST_EXACT, antialias=True),
    ]),
    
    transforms.RandomApply(  
        [transforms.ColorJitter(
            brightness=(0.8, 1.2),  # Brightness range: [0.8, 1.2]
            contrast=(0.8, 1.2)     # Contrast range: [0.8, 1.2]
        )],
        p=0.05                      # 5% of probability
    ),
    
    transforms.RandomHorizontalFlip(p=0.05),  
    
    transforms.RandomApply( 
        # Rotation range: [3 - 45]
        [transforms.RandomRotation(degrees=(3, 45))],
        p=0.05
    ),
    transforms.ToTensor()
])

augmentation_pipeline_val = transforms.Compose([
    transforms.Resize((224, 224), interpolation=InterpolationMode.BILINEAR, antialias=False),
    transforms.ToTensor()
])

In [4]:
df = pd.read_csv('wiki_imdb_lfw.csv')
df.shape

(536284, 8)

In [5]:
df = df[df.include == True].reset_index(drop=True)
df.dropna(inplace=True)
df.shape

(528209, 8)

In [6]:
train = df[df.split == 'train'].reset_index(drop=True)
val = df[df.split == 'val'].reset_index(drop=True)
test = df[df.split == 'test'].reset_index(drop=True)

In [7]:
# train = train.iloc[:50, :]
# val = val.iloc[:50, :]

In [8]:
train.shape, val.shape, test.shape

((414569, 8), (100407, 8), (13233, 8))

In [9]:
with open('encoder.json', 'r', encoding='utf-8') as f:
    encoder = json.load(f)

with open('decoder.json', 'r', encoding='utf-8') as f:
    decoder = json.load(f)

In [10]:
num_classes = len(encoder)
num_classes

81272

In [11]:
class dataset(Dataset):
    def __init__(self, df, augmentation_pipeline, device):
        super(dataset, self).__init__()
        self.df = df
        self.device = device
        self.augmentation_pipeline = augmentation_pipeline
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, x):
        path = self.df.path[x]
        name = self.df.name[x]
        name = encoder[name]
        name = torch.tensor([name], dtype=torch.int64, device=self.device)
        
        img = Image.open(path).convert('RGB')
        img = self.augmentation_pipeline(img)
        img = img.to(self.device)
        return img, name

In [12]:
img, label = next(iter(dataset(df, augmentation_pipeline_train, device='cuda')))
img.shape, label.shape

C:\Users\yerda\AppData\Roaming\Python\Python311\site-packages\torchvision\transforms\functional.py:488: UserWarning: Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.
  warnings.warn("Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.")


(torch.Size([3, 224, 224]), torch.Size([1]))

In [13]:
trainLoader = DataLoader(dataset(train, augmentation_pipeline_train, device='cuda'), batch_size=64, shuffle=True)
valLoader = DataLoader(dataset(val, augmentation_pipeline_val, device='cuda'), batch_size=64, shuffle=False)
testLoader = DataLoader(dataset(test, augmentation_pipeline_val, device='cuda'), batch_size=64, shuffle=False)

# Model

In [15]:
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
mobilenet.classifier[1] = torch.nn.Linear(in_features=1280, out_features=128)
mobilenet = mobilenet.cuda()

# ArcFace

In [17]:
class ArcFaceLoss(nn.Module):
    def __init__(self, dimention, num_classes, s=64.0, m=0.50):
        super(ArcFaceLoss, self).__init__()
        self.s = s  # Feature scale (often set to 64.0)
        self.m = m  # Angular margin (often set to 0.50)
        
        # Initialize learnable weight parameters
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, dimention))
        nn.init.xavier_uniform_(self.weight)  # Xavier initialization

    def forward(self, input, labels):
        # Normalize input features and weight vectors to unit vectors
        cosine = torch.matmul(F.normalize(input), F.normalize(self.weight).T)
        
        # Clip cosine values for numerical stability
        cosine = cosine.clamp(-1.0, 1.0)
        
        # Add angular margin directly to the cosine value (no need to compute angle)
        target_cosine = cosine.gather(1, labels.view(-1, 1))  # Cosine value for the correct class
        cosine_with_margin = target_cosine + self.m  # Add margin
        
        # Clip the margin-modified cosine for numerical stability
        cosine_with_margin = cosine_with_margin.clamp(-1.0, 1.0)
        
        # Create a one-hot encoding for the target class labels
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)
        
        # Combine margin cosine for target class, and regular cosine for others
        logits = (one_hot * cosine_with_margin) + ((1.0 - one_hot) * cosine)
        
        # Scale logits by the factor 's'
        logits *= self.s
        
        # Compute cross-entropy loss
        loss = F.cross_entropy(logits, labels)
        return loss

# Training

In [19]:
criterion = ArcFaceLoss(dimention=128, num_classes=num_classes, s=64.0, m=0.50).cuda()
optimization = torch.optim.Adam(mobilenet.parameters(), lr=1e-3)

In [20]:
os.makedirs('models/', exist_ok=True)

def model_checkpoint_tracer(model, loss_function, global_min_val_loss, val_loss, train_loss, epoch) -> None:
    if val_loss < global_min_val_loss:
        global_min_val_loss = val_loss
        checkpoint_path = f'models/best_model_epoch_{epoch}_val_loss_{val_loss:.4f}.pt'
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'loss_function_state_dict': loss_function.state_dict(),
            'val_loss': val_loss,
            'train_loss': train_loss
        }, checkpoint_path)
        print(f"The best model with min val loss of {val_loss} was saved at iteration of {epoch}!")
        global_min_val_loss = val_loss
    checkpoint_path = f'models/last_model_epoch_{epoch}_val_loss_{val_loss:.4f}.pt'
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'loss_function_state_dict': loss_function.state_dict(),
        'val_loss': val_loss,
        'train_loss': train_loss
    }, checkpoint_path)
    return global_min_val_loss

In [ ]:
num_iteration = 1000
global_min_val_loss = torch.inf
train_loss_list, val_loss_list = [], []

for epoch in tqdm.trange(num_iteration):
    mobilenet.train()
    epoch_train_loss = 0 
    count = 0
    for images, targets in trainLoader:
        optimization.zero_grad()
        logits = mobilenet(images)
        loss = criterion(logits, targets.squeeze(1)) 
        loss.backward()
        optimization.step()
        b_size = images.shape[0]
        epoch_train_loss += loss.item()
        count += 1

    epoch_train_loss /= count  # Average loss per epoch
    train_loss_list.append(epoch_train_loss)

    mobilenet.eval()
    epoch_val_loss = 0 
    count = 0
    with torch.no_grad():
        for images, targets in valLoader:
            logits = mobilenet(images)
            loss = criterion(logits, targets.squeeze(1))
            b_size = images.shape[0]
            epoch_val_loss += loss.item()
            count += 1

    epoch_val_loss /= count
    val_loss_list.append(epoch_val_loss)  
    
    print(f"Epoch {epoch }/{num_iteration} - Train Loss: {epoch_train_loss:.8f}, Val Loss: {epoch_val_loss:.8f}")
    
    global_min_val_loss = model_checkpoint_tracer(
        mobilenet, criterion, global_min_val_loss, epoch_val_loss, epoch_train_loss, epoch
    )

  0%|                                                                                         | 0/1000 [00:00<?, ?it/s]C:\Users\yerda\AppData\Roaming\Python\Python311\site-packages\torchvision\transforms\functional.py:488: UserWarning: Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.
  warnings.warn("Anti-alias option is always applied for PIL Image input. Argument antialias is ignored.")


Epoch 0/1000 - Train Loss: 0.03540979, Val Loss: 0.08990408
The best model with min val loss of 0.08990408274730602 was saved at iteration of 0!


  0%|                                                                        | 1/1000 [1:36:12<1601:53:46, 5772.60s/it]

Epoch 1/1000 - Train Loss: 0.00868080, Val Loss: 0.00132904
The best model with min val loss of 0.0013290422242813234 was saved at iteration of 1!


  0%|▏                                                                       | 2/1000 [3:07:47<1554:59:38, 5609.20s/it]

...

Epoch 9/1000 - Train Loss: 0.00072775, Val Loss: 0.00127574


  1%|▊                                                                     | 11/1000 [16:52:37<1511:25:01, 5501.62s/it]

Epoch 10/1000 - Train Loss: 0.00072047, Val Loss: 0.00131576


  1%|▊                                                                     | 12/1000 [18:23:41<1506:42:54, 5490.06s/it]

Epoch 11/1000 - Train Loss: 0.00071352, Val Loss: 0.00131173
